# 4.1 Train XGBoost with SageMaker Debugger (run in SageMaker Studio)

Built-in SageMaker XGBoost 1.7-1, `multi:softprob`, fixed hyperparameters (no HPO), unweighted, early stopping on validation. One `ml.m5.xlarge`. Debugger rules: LossNotDecreasing, Overfit, ClassImbalance + the XGBoost report. The model is registered in the Model Registry as `PendingManualApproval`.

In [ ]:
# SageMaker Python SDK v3 (default on current Studio images) removed the v2 modules this notebook uses.
# Pin v2; on first run this installs it and restarts the kernel - then run the notebook again from the top.
from importlib.metadata import version, PackageNotFoundError
try:
    _sm_ver = version("sagemaker")
except PackageNotFoundError:
    _sm_ver = "0"
if int(_sm_ver.split(".")[0]) != 2:
    %pip uninstall -q -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
    %pip install -q "sagemaker>=2.200,<3"
    print(f"Replaced sagemaker {_sm_ver} with v2 - restarting kernel; re-run the notebook from the top.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("sagemaker", _sm_ver)

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import io, json
import boto3, numpy as np, pandas as pd
from src.pamap2.config import load_config, s3_uri
from src.pamap2.schema import ACTIVITY_NAMES
from src.pamap2 import evaluation as E
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
s3 = session.client("s3")
SPLITS = s3_uri(cfg, "splits")
REPORTS = s3_uri(cfg, "models", "reports/")

def split_key(uri):
    b, k = uri[len("s3://"):].split("/", 1)
    return b, k
def read_parquet(name):
    b, k = split_key(SPLITS + name)
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=b, Key=k)["Body"].read()))
def read_json(uri):
    b, k = split_key(uri)
    return json.loads(s3.get_object(Bucket=b, Key=k)["Body"].read())
def write_json(uri, obj):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=json.dumps(obj, indent=2).encode())
    print("Wrote", uri)

FEATURES = read_json(SPLITS + "feature_columns.json")
LABEL_MAP = E.load_label_map(read_json(SPLITS + "label_map.json"))   # class index -> activity_id
K = len(LABEL_MAP)
CLASSES = list(range(K))
def class_name(i):
    return ACTIVITY_NAMES[LABEL_MAP[int(i)]]
print(K, "classes;", len(FEATURES), "features")


In [ ]:
import sagemaker
from sagemaker.inputs import TrainingInput
from sagemaker.debugger import Rule, rule_configs, DebuggerHookConfig, CollectionConfig
sm = session.client("sagemaker")
sm_session = sagemaker.Session(boto_session=session)
ROLE = cfg["role_arn"]
image = sagemaker.image_uris.retrieve("xgboost", cfg["region"], cfg["xgboost_version"])
print(image)

## Estimator, hyperparameters, Debugger

In [ ]:
output_path = s3_uri(cfg, "models", "xgboost/")
estimator = sagemaker.estimator.Estimator(
    image_uri=image, role=ROLE, instance_count=1, instance_type="ml.m5.xlarge",
    output_path=output_path, sagemaker_session=sm_session,
    base_job_name=f"{cfg['prefix']}-xgboost",
    max_run=3600,
    debugger_hook_config=DebuggerHookConfig(
        s3_output_path=s3_uri(cfg, "models", "xgboost/debugger/"),
        collection_configs=[CollectionConfig(name=n, parameters={"save_interval": "5"})
                            for n in ("metrics", "feature_importance", "average_shap",
                                      "labels", "predictions")]),
    rules=[Rule.sagemaker(rule_configs.loss_not_decreasing()),
           Rule.sagemaker(rule_configs.overfit()),
           Rule.sagemaker(rule_configs.class_imbalance()),
           Rule.sagemaker(rule_configs.create_xgboost_report())],
)
estimator.set_hyperparameters(
    objective="multi:softprob", num_class=K, num_round=500, early_stopping_rounds=20,
    eval_metric="mlogloss,merror", max_depth=6, eta=0.1, subsample=0.8, colsample_bytree=0.8)

## Train (billable: ~one ml.m5.xlarge job)

In [ ]:
estimator.fit({"train": TrainingInput(SPLITS + "train.csv", content_type="text/csv"),
               "validation": TrainingInput(SPLITS + "validation.csv", content_type="text/csv")}, wait=True)
job = estimator.latest_training_job.name
print("Training job:", job, "| model:", estimator.model_data)

## Debugger rule statuses and report

In [ ]:
summary = estimator.latest_training_job.rule_job_summary()
display(pd.DataFrame([{k: r.get(k) for k in ("RuleConfigurationName", "RuleEvaluationStatus", "StatusDetails")}
                      for r in summary]))
print("Debugger artifacts:", estimator.latest_job_debugger_artifacts_path())
print("XGBoost report:", estimator.latest_job_debugger_artifacts_path() + "rule-output/CreateXgboostReport/")

## Register in Model Registry (PendingManualApproval)

In [ ]:
group = cfg["model_package_group"]
try:
    sm.describe_model_package_group(ModelPackageGroupName=group)
except sm.exceptions.ClientError as e:
    if "does not exist" not in str(e) and "ValidationException" not in str(e):
        raise
    sm.create_model_package_group(ModelPackageGroupName=group,
                                  ModelPackageGroupDescription="PAMAP2 HAR XGBoost models")
pkg = estimator.register(content_types=["text/csv"], response_types=["text/csv"],
                         inference_instances=["ml.m5.large"], transform_instances=["ml.m5.large"],
                         model_package_group_name=group, approval_status="PendingManualApproval",
                         description="PAMAP2 activity recognition, multi:softprob")
pkg_arn = pkg.model_package_arn
final = {m["MetricName"]: m["Value"] for m in sm.describe_training_job(TrainingJobName=job).get("FinalMetricDataList", [])}
print(final)
sm.update_model_package(ModelPackageArn=pkg_arn, CustomerMetadataProperties={
    **{k.replace(":", "_"): f"{v:.6f}" for k, v in final.items()}, "training_job": job})
print("Registered:", pkg_arn)

In [ ]:
write_json(REPORTS + "xgboost_training.json", {
    "training_job": job, "model_data": estimator.model_data, "image_uri": image,
    "model_package_arn": pkg_arn, "final_metrics": final,
    "debugger_artifacts": estimator.latest_job_debugger_artifacts_path()})